# Validación general Nacional vs. Regional (multi-parámetro)

Compara cualquier combinación de `Parameter` / `TECHNOLOGY` / `FUEL` entre:

- **Nacional**: `SAND_Nacional_base/01-04-2026 SAND BASE v10.xlsx`
- **Regional**: `SAND_Regional/scenario_23_Parameters_SAND.xlsx` (7 regiones, prefijo `PREFIJO_` en `FUEL` y `TECHNOLOGY`)

Consolida el Regional sumando las 7 regiones (equivalente nacional) y lo cruza contra el Nacional para detectar diferencias.

## 0. Configuración de filtros

Edita estas listas para acotar la validación. Deja la lista vacía `[]` si no deseas filtrar por ese criterio.

In [ ]:
# Deja la lista vacía [] si no deseas filtrar por ese criterio
PARAMETROS_A_FILTRAR = ['AccumulatedAnnualDemand', 'TotalTechnologyAnnualActivityLowerLimit']
TECNOLOGIAS_A_FILTRAR = []
FUELS_A_FILTRAR = ['TRABUS', 'TRAAVI']

# Modo de coincidencia para TECNOLOGIAS_A_FILTRAR y FUELS_A_FILTRAR:
#   'exacto'   -> el valor debe coincidir exactamente (ej. 'TRABUS')
#   'contiene' -> basta con que contenga la subcadena (ej. 'TRA' trae TRABUS, TRAAVI, ...)
MODO_FILTRO = 'exacto'

# Parámetros de VALOR FIJO (intensivos): NO se suman entre regiones.
# Cada región debe conservar exactamente el valor nacional (ratios, factores,
# costos unitarios, vidas útiles...). La validación se hace región por región.
# Todo parámetro que NO esté en esta lista se valida como aditivo (Nacional = suma de regiones).
PARAMETROS_VALOR_FIJO = [
    'InputActivityRatio',
    'OutputActivityRatio',
    'EmissionActivityRatio',
    'CapacityFactor',
    'AvailabilityFactor',
    'CapacityToActivityUnit',
    'CapitalCost',
    'FixedCost',
    'VariableCost',
    'OperationalLife',
]

In [ ]:
import re
import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.6f}')

In [ ]:
# --- Rutas de archivos ---
NACIONAL_PATH = "SAND_Nacional_base/01-04-2026 SAND BASE v10.xlsx"
REGIONAL_PATH = "SAND_Regional/Escenario_SAND_Regional.xlsx"

REGIONES_VALIDAS = {"CA", "OR", "SO", "AN", "NE", "SE", "IN"}
TOLERANCIA = 1e-4

## 1. Carga de archivos

In [ ]:
df_nacional_raw = pd.read_excel(NACIONAL_PATH, sheet_name="Parameters")
df_regional_raw = pd.read_excel(REGIONAL_PATH, sheet_name="Parameters")

YEAR_COLS = [c for c in df_nacional_raw.columns if re.fullmatch(r"\d{4}", str(c))]
print(f"Nacional: {df_nacional_raw.shape}  |  Regional: {df_regional_raw.shape}")
print(f"{len(YEAR_COLS)} columnas de año: {YEAR_COLS[0]}..{YEAR_COLS[-1]}")

## 2. Consolidación Regional: limpiar prefijos, identificar región y agrupar

Se limpia el prefijo `REGION_` de `FUEL`/`TECHNOLOGY` y se guarda la región en una columna aparte. Luego se preparan **dos** vistas del Regional:

- **Aditiva** (`df_reg_grouped`): suma de las 7 regiones, para parámetros de cantidad (demandas, límites de actividad...).
- **Por región** (`df_reg_por_region`): valor individual de cada región, para los `PARAMETROS_VALOR_FIJO` (ratios, factores, costos...), donde cada región debe igualar el valor nacional.

In [ ]:
def quitar_prefijo_region(valor):
    """Quita el prefijo REGION_ (ej. CA_TRABUS -> TRABUS). Deja igual si no aplica."""
    if pd.isna(valor):
        return valor
    texto = str(valor)
    prefijo, sep, resto = texto.partition("_")
    if sep and prefijo in REGIONES_VALIDAS:
        return resto
    return texto


def extraer_region(valor):
    """Devuelve la región del prefijo (ej. CA_TRABUS -> CA) o None si no tiene."""
    if pd.isna(valor):
        return None
    prefijo, sep, _ = str(valor).partition("_")
    if sep and prefijo in REGIONES_VALIDAS:
        return prefijo
    return None


df_reg_clean = df_regional_raw.copy()
# La región se toma del prefijo de TECHNOLOGY y, si no tiene, del de FUEL
df_reg_clean["Region"] = df_reg_clean["TECHNOLOGY"].apply(extraer_region)
df_reg_clean["Region"] = df_reg_clean["Region"].fillna(df_reg_clean["FUEL"].apply(extraer_region))
df_reg_clean["FUEL"] = df_reg_clean["FUEL"].apply(quitar_prefijo_region)
df_reg_clean["TECHNOLOGY"] = df_reg_clean["TECHNOLOGY"].apply(quitar_prefijo_region)

df_reg_clean[["Parameter", "Region", "TECHNOLOGY", "FUEL"]].drop_duplicates().head()

In [ ]:
# Formato largo con la región identificada
ID_COLS = ["Parameter", "TECHNOLOGY", "FUEL"]

df_reg_long = df_reg_clean[ID_COLS + ["Region"] + YEAR_COLS].melt(
    id_vars=ID_COLS + ["Region"], value_vars=YEAR_COLS, var_name="YEAR", value_name="VALUE"
)
df_reg_long["YEAR"] = df_reg_long["YEAR"].astype(int)
df_reg_long["VALUE"] = pd.to_numeric(df_reg_long["VALUE"], errors="coerce").fillna(0)

# Vista ADITIVA: suma de las 7 regiones por Parameter, TECHNOLOGY, FUEL, YEAR
df_reg_grouped = (
    df_reg_long.groupby(ID_COLS + ["YEAR"], dropna=False, as_index=False)["VALUE"]
    .sum()
    .rename(columns={"VALUE": "Suma_Regional"})
)

# Vista POR REGIÓN: valor individual de cada región (para parámetros de valor fijo)
df_reg_por_region = (
    df_reg_long.groupby(ID_COLS + ["Region", "YEAR"], dropna=False, as_index=False)["VALUE"]
    .sum()
    .rename(columns={"VALUE": "Valor_Region"})
)

df_reg_grouped.head()

## 3. Filtrado múltiple del Nacional

In [ ]:
def aplicar_filtro(df, columna, valores, modo="exacto"):
    """Filtra la columna según la lista de valores; si la lista está vacía, no filtra.

    modo='exacto'   -> coincidencia exacta (.isin)
    modo='contiene' -> basta con que el valor contenga alguna de las subcadenas
    """
    if not valores:
        return df
    if modo == "contiene":
        patron = "|".join(re.escape(v) for v in valores)
        return df[df[columna].astype(str).str.contains(patron, na=False)]
    return df[df[columna].isin(valores)]


df_nac_filt = df_nacional_raw.copy()
# El filtro de Parameter siempre es exacto; el modo aplica a TECHNOLOGY y FUEL
df_nac_filt = aplicar_filtro(df_nac_filt, "Parameter", PARAMETROS_A_FILTRAR)
df_nac_filt = aplicar_filtro(df_nac_filt, "TECHNOLOGY", TECNOLOGIAS_A_FILTRAR, MODO_FILTRO)
df_nac_filt = aplicar_filtro(df_nac_filt, "FUEL", FUELS_A_FILTRAR, MODO_FILTRO)

print(f"Filas nacionales tras filtrar: {len(df_nac_filt)} de {len(df_nacional_raw)}")
print("Parámetros presentes:", sorted(df_nac_filt['Parameter'].unique()))

In [ ]:
df_nac_long = df_nac_filt[ID_COLS + YEAR_COLS].melt(
    id_vars=ID_COLS, value_vars=YEAR_COLS, var_name="YEAR", value_name="Valor_Nacional"
)
df_nac_long["YEAR"] = df_nac_long["YEAR"].astype(int)
df_nac_long["Valor_Nacional"] = pd.to_numeric(df_nac_long["Valor_Nacional"], errors="coerce").fillna(0)

df_nac_long.head()

## 4. Cruce (merge) Nacional vs. Regional

El cruce se hace en dos rutas según el tipo de parámetro y luego se unifica:

- **Aditivos**: `Valor_Nacional` vs. **suma** de las regiones (una fila por combinación; `Region = 'SUMA_REGIONES'`).
- **Valor fijo** (`PARAMETROS_VALOR_FIJO`): `Valor_Nacional` vs. el valor de **cada región** (una fila por región; la columna `Region` indica cuál falló).

In [ ]:
# NaN no coincide con NaN al hacer merge: se normalizan las llaves con un marcador
# para que TECHNOLOGY/FUEL vacíos (según el parámetro) sí crucen correctamente.
MARCADOR_VACIO = "__SIN_VALOR__"
LLAVES_MERGE = ["Parameter", "TECHNOLOGY", "FUEL", "YEAR"]


def normalizar_llaves(df):
    df = df.copy()
    df["TECHNOLOGY"] = df["TECHNOLOGY"].fillna(MARCADOR_VACIO)
    df["FUEL"] = df["FUEL"].fillna(MARCADOR_VACIO)
    return df


df_nac_merge = normalizar_llaves(df_nac_long)
es_valor_fijo = df_nac_merge["Parameter"].isin(PARAMETROS_VALOR_FIJO)

# --- Ruta 1: parámetros ADITIVOS -> comparar contra la suma de las regiones ---
df_reg_suma = normalizar_llaves(df_reg_grouped)
df_comp_aditivo = pd.merge(
    df_nac_merge[~es_valor_fijo], df_reg_suma, on=LLAVES_MERGE, how="left"
).rename(columns={"Suma_Regional": "Valor_Regional"})
df_comp_aditivo["Region"] = "SUMA_REGIONES"

# --- Ruta 2: parámetros de VALOR FIJO -> comparar contra cada región por separado ---
df_reg_fijo = normalizar_llaves(df_reg_por_region)
df_reg_fijo = df_reg_fijo[df_reg_fijo["Parameter"].isin(PARAMETROS_VALOR_FIJO)]
# Filas regionales cuyo TECHNOLOGY/FUEL no traía prefijo de región
df_reg_fijo["Region"] = df_reg_fijo["Region"].fillna("SIN_PREFIJO")

df_comp_fijo = pd.merge(
    df_nac_merge[es_valor_fijo],
    df_reg_fijo[LLAVES_MERGE + ["Region", "Valor_Region"]],
    on=LLAVES_MERGE,
    how="left",
).rename(columns={"Valor_Region": "Valor_Regional"})
# Combinaciones nacionales que no cruzaron con ninguna fila regional
df_comp_fijo["Region"] = df_comp_fijo["Region"].fillna("SIN_DATO_REGIONAL")

# --- Unificación de ambas rutas ---
df_comparacion = pd.concat([df_comp_aditivo, df_comp_fijo], ignore_index=True)

# Manejo de nulos: combinaciones sin dato regional -> 0
df_comparacion["Valor_Nacional"] = df_comparacion["Valor_Nacional"].fillna(0)
df_comparacion["Valor_Regional"] = df_comparacion["Valor_Regional"].fillna(0)

df_comparacion["Diferencia"] = df_comparacion["Valor_Nacional"] - df_comparacion["Valor_Regional"]
df_comparacion["Diferencia_Abs"] = df_comparacion["Diferencia"].abs()

# Restaurar el marcador de vacío a NaN para una lectura más limpia
df_comparacion["TECHNOLOGY"] = df_comparacion["TECHNOLOGY"].replace(MARCADOR_VACIO, pd.NA)
df_comparacion["FUEL"] = df_comparacion["FUEL"].replace(MARCADOR_VACIO, pd.NA)

df_comparacion = df_comparacion.sort_values(
    ["Parameter", "TECHNOLOGY", "FUEL", "YEAR", "Region"]
).reset_index(drop=True)

## 5. Resultado final

In [ ]:
columnas_salida = [
    "Parameter", "TECHNOLOGY", "FUEL", "YEAR", "Region",
    "Valor_Nacional", "Valor_Regional", "Diferencia",
]
df_reporte = df_comparacion[columnas_salida]

print(f"Filas del reporte: {len(df_reporte)}")
df_reporte

## 6. Detección de discrepancias

Se aíslan únicamente las filas con diferencia real (`abs(Diferencia) > TOLERANCIA`) para evitar falsos positivos por punto flotante de Excel.

In [ ]:
# DataFrame exclusivo de errores: solo filas cuya diferencia supera la tolerancia
NOMBRES_REPORTE = {
    "Parameter": "Parámetro",
    "TECHNOLOGY": "Tecnología",
    "FUEL": "Fuel",
    "YEAR": "Año",
    "Region": "Región",
    "Valor_Nacional": "Valor Nacional",
    "Valor_Regional": "Valor Regional",
    "Diferencia": "Diferencia",
}

df_discrepancias = (
    df_comparacion[df_comparacion["Diferencia_Abs"] > TOLERANCIA][columnas_salida]
    .rename(columns=NOMBRES_REPORTE)
    .sort_values(["Parámetro", "Tecnología", "Fuel", "Año", "Región"])
    .reset_index(drop=True)
)

if not df_discrepancias.empty:
    print(f"ALERTA: {len(df_discrepancias)} de {len(df_comparacion)} combinaciones superan la tolerancia ({TOLERANCIA}).")
    print(f"Diferencia máxima observada: {df_discrepancias['Diferencia'].abs().max():,.6f}")
    resumen = df_discrepancias.groupby("Parámetro").size().rename("Filas con diferencia")
    print("\nDiscrepancias por parámetro:")
    print(resumen.to_string())
    # Para parámetros de valor fijo, muestra en qué regiones se concentran los errores
    fijas = df_discrepancias[df_discrepancias["Parámetro"].isin(PARAMETROS_VALOR_FIJO)]
    if not fijas.empty:
        print("\nDiscrepancias de valor fijo por región:")
        print(fijas.groupby("Región").size().rename("Filas").to_string())
else:
    print(f"OK: todas las diferencias están dentro de la tolerancia ({TOLERANCIA}).")

df_discrepancias

## 7. Exportar reporte a Excel

Genera un `.xlsx` con dos hojas:

- **Discrepancias**: solo los registros con diferencia real (respetando la tolerancia).
- **Comparacion_Total**: todos los datos que pasaron el filtro inicial, como respaldo.

In [ ]:
ARCHIVO_SALIDA = "Reporte_Validacion_Nacional_vs_Regional.xlsx"

# Hoja de respaldo con los mismos nombres de columna que la hoja de discrepancias
df_comparacion_total = df_reporte.rename(columns=NOMBRES_REPORTE)

with pd.ExcelWriter(ARCHIVO_SALIDA, engine="openpyxl") as writer:
    df_discrepancias.to_excel(writer, sheet_name="Discrepancias", index=False)
    df_comparacion_total.to_excel(writer, sheet_name="Comparacion_Total", index=False)

print(f"Reporte exportado: {ARCHIVO_SALIDA}")
print(f"  - Hoja 'Discrepancias':     {len(df_discrepancias)} filas")
print(f"  - Hoja 'Comparacion_Total': {len(df_comparacion_total)} filas")